# Item 10 — a comparação única

Cinco modelos, as mesmas células, o mesmo eixo semanal:

| modelo | o que é |
|---|---|
| `persistencia` | ŷ(w+k) = y(w). O piso que qualquer modelo precisa bater |
| `sir` | SIR clássico reajustado causalmente em cada origem (Oliveira et al., BraSNAM 2025) |
| `sem_grafo` | o preditor neural sem acesso a nenhuma relação (item 07) |
| `embaralhado` | a mesma GNN sobre topologia aleatória de mesmo grau (item 08) |
| `proposta` | MusicDiffusionGNN (item 06) |

Dois regimes de split, dois recortes, três horizontes, dois regimes de chart. A estatística é a do
item 03: erro pareado **por música**, IC bootstrap sobre a diferença, Wilcoxon e correção de Holm.

Este é o notebook em que o pré-compromisso do
[ADR-0001](../docs/adr/0001-precompromisso-de-falseamento.md) é acionado. A regra de leitura foi
registrada antes de qualquer número, e a última seção apenas a aplica ao que saiu.

Entrada: a pasta `avaliacao_matriz/` produzida por `item09_avaliacao_matriz_colab.ipynb`, incluindo
`origens_{regime}.parquet`. SIR e persistência são refeitos **sobre essas mesmas origens**: sem
isso a tabela não é comparável célula a célula.


## 0. Ambiente — Colab (GPU) ou local

No Colab, clona o repositório (código + artefatos de dados versionados) e instala o PyG.
Ative a GPU em *Ambiente de execução → Alterar tipo de runtime → GPU*.
Repo privado: cole um PAT em `GITHUB_TOKEN`.

> Os grafos `hetero_full_current.pt` e `hetero_full_pre_pandemia.pt` precisam estar **commitados**
> antes de clonar: os CSVs brutos da rede de gêneros não são versionados, então o grafo não pode
> ser reconstruído aqui dentro.


In [ ]:
import sys, os, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

REPO_URL     = "https://github.com/cristianomendieta/music-influence-gnn.git"
REPO_BRANCH  = "main"
REPO_DIR     = "/content/music-influence-gnn"
GITHUB_TOKEN = ""   # repo privado: cole um PAT aqui OU defina a env GITHUB_TOKEN

DATA_FILES = [
    "data/processed/graph/hetero_full_current.pt",
    "data/processed/graph/hetero_full_pre_pandemia.pt",
    "data/processed/graph/node_id_map.json",
    "data/processed/timeseries.parquet",
]

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, REPO_DIR])

if IS_COLAB:
    if Path(REPO_DIR, "pyproject.toml").exists():
        # O runtime sobrevive ao restart do kernel: um clone de sessão anterior ficaria
        # para trás em silêncio e o import viria do código velho. Sincroniza sempre.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
    else:
        tok = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN", "")
        url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
        print(f"Clonando {REPO_URL} (branch {REPO_BRANCH})...")
        if _clone(url).returncode != 0:
            from getpass import getpass
            tok = getpass("Clone falhou (repo privado?). Cole um GitHub token (PAT): ")
            _clone(REPO_URL.replace("https://", f"https://{tok}@")).check_returncode()
    os.chdir(REPO_DIR)
    print("commit em uso:", subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                                           capture_output=True, text=True).stdout.strip())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-geometric", "pyarrow"], check=True)
    faltando = [f for f in DATA_FILES if not Path(REPO_DIR, f).exists()]
    print("✓ Colab pronto. cwd =", os.getcwd())
    if faltando:
        print("⚠️ FALTAM no repositório:", faltando)
else:
    print("Local — nada a clonar.")


In [ ]:
import json, time, shutil
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

_anchor = Path(globals().get("__vsc_ipynb_file__", os.getcwd()))
if not _anchor.exists():
    _anchor = Path(os.getcwd())
ROOT = _anchor if _anchor.is_dir() else _anchor.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), f"raiz do projeto não encontrada a partir de {_anchor}"
sys.path.insert(0, str(ROOT / "src"))

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 10
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

from music_diffusion_gnn.evaluation.sir_eval import run_sir_mode2
from music_diffusion_gnn.evaluation.stats import (
    aggregate_seeds, directional_accuracy, holm_correction, paired_by_song,
)
from music_diffusion_gnn.models.baselines import persistence_multistep
from music_diffusion_gnn.training.dataset import aggregate_weekly

GRAPH_DIR = ROOT / "data" / "processed" / "graph"
NMAP_PATH = GRAPH_DIR / "node_id_map.json"
TS_PATH   = ROOT / "data" / "processed" / "timeseries.parquet"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if IS_COLAB and DEVICE != "cuda":
    raise RuntimeError("GPU não ativada no Colab: Ambiente de execução → Alterar tipo de runtime → GPU.")
print("ROOT =", ROOT, "| DEVICE =", DEVICE)


In [ ]:
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/music-influence-gnn")
else:
    BASE = ROOT / "results"
MATRIZ = BASE / "avaliacao_matriz"
OUT    = BASE / "comparacao_final"
OUT.mkdir(parents=True, exist_ok=True)

KS       = (1, 2, 4)
REGIMES  = ["current", "pre_pandemia"]
RECORTES = ["onchart", "full"]        # on-chart é a leitura principal (ADR-0004)
MODELOS  = ["persistencia", "sir", "sem_grafo", "embaralhado", "proposta"]
SEED_BOOTSTRAP = 42
ALFA = 0.05

print("matriz ←", MATRIZ, f"({len(list(MATRIZ.glob('mode2_*.parquet')))} parquets de Modo 2)")
print("saída  →", OUT)


## 1. As predições da escada neural

Só a leitura de orçamento `completo` entra na comparação principal: é a que avalia sobre a
topologia real inteira. A leitura de orçamento casado fica reportada no notebook do item 09, como
o ticket 08 exige, e não se mistura com esta tabela.

As três seeds de cada modelo são reduzidas a uma predição por origem pela **média entre seeds**.
Isso mantém o pareamento por origem que o teste estatístico exige e trata a seed como réplica do
mesmo modelo, não como modelo diferente. A dispersão entre seeds continua reportada, separada, na
tabela de RMSE.


In [ ]:
arquivos = sorted(MATRIZ.glob("mode2_*_completo.parquet"))
assert arquivos, f"nenhum mode2_*_completo.parquet em {MATRIZ}"
neural = pd.concat([pd.read_parquet(f) for f in arquivos], ignore_index=True)

CHAVE_ORIGEM = ["song_id", "chart", "origin_week", "k", "split_regime"]
neural_medio = (neural.groupby(CHAVE_ORIGEM + ["variante"], observed=True)
                      .agg(y_true=("y_true", "first"), y_pred=("y_pred", "mean"),
                           onchart=("onchart", "first"), n_seeds=("seed", "nunique"))
                      .reset_index()
                      .rename(columns={"variante": "modelo"}))
print(neural_medio.groupby(["modelo", "split_regime"]).agg(
    origens=("y_pred", "size"), seeds=("n_seeds", "min")))


## 2. SIR e persistência sobre as mesmas origens

O SIR é reajustado do zero em cada origem, com dados até ela e projeção para `w+k`: é o único
regime em que a comparação com um modelo causal é justa. É o passo mais caro deste notebook
(milhares de ajustes), então fica em cache por regime.

Origens em que o ajuste não converge entram com `y_pred` nulo e `converged=False`, nunca são
descartadas em silêncio. A contagem é reportada junto da tabela.


In [ ]:
ts = pd.read_parquet(TS_PATH)
weekly = aggregate_weekly(ts)

def origens(regime: str) -> pd.DataFrame:
    p = MATRIZ / f"origens_{regime}.parquet"
    assert p.exists(), f"{p} não existe — rode o notebook do item 09 primeiro"
    return pd.read_parquet(p)

partes = []
for regime in REGIMES:
    if not (MATRIZ / f"origens_{regime}.parquet").exists():
        continue
    o = origens(regime)

    p_sir = OUT / f"sir_mode2_{regime}.parquet"
    if p_sir.exists():
        sir = pd.read_parquet(p_sir)
        print(f"{regime}: SIR em cache ({len(sir):,} linhas)")
    else:
        t0 = time.time()
        sir = run_sir_mode2(ts, o, ks=KS, regime=regime)
        sir.to_parquet(p_sir, index=False)
        print(f"{regime}: SIR {len(sir):,} linhas, "
              f"{int((~sir.converged).sum())} não convergiram [{(time.time()-t0)/60:.1f} min]")
    sir = sir.rename(columns={"y_pred_sir": "y_pred"})
    sir["modelo"], sir["split_regime"] = "sir", regime
    partes.append(sir)

    pers = persistence_multistep(weekly, o, ks=KS)
    pers["modelo"], pers["split_regime"], pers["converged"] = "persistencia", regime, True
    partes.append(pers)

classicos = pd.concat(partes, ignore_index=True)

# y_true e a marca de recorte vêm da tabela neural, para que as três fontes falem da mesma célula
ref = neural_medio[CHAVE_ORIGEM + ["y_true", "onchart"]].drop_duplicates(CHAVE_ORIGEM)
classicos = classicos.merge(ref, on=CHAVE_ORIGEM, how="inner")

tudo = pd.concat([neural_medio.assign(converged=True), classicos], ignore_index=True)
tudo = tudo[CHAVE_ORIGEM + ["modelo", "y_true", "y_pred", "onchart", "converged"]]
tudo.to_parquet(OUT / "predicoes_cinco_modelos.parquet", index=False)
print("\n", tudo.groupby(["modelo", "split_regime"]).size())


## 3. A tabela

RMSE e acerto direcional por célula. Para os três modelos neurais, o desvio entre seeds vem da
tabela do item 09 (`resumo_mode2.parquet`), porque aqui as seeds já foram promediadas; para SIR e
persistência não há seed, e o desvio é zero por construção.


In [ ]:
y_origem = weekly.set_index(["song_id", "chart", "week"])["y_week"]
tudo["y_origem"] = [
    float(y_origem.get((s, c, int(w)), np.nan))
    for s, c, w in zip(tudo.song_id, tudo.chart, tudo.origin_week)
]

def celulas(df, recorte):
    sub = df if recorte == "full" else df[df.onchart]
    linhas = []
    for (regime, modelo, chart, k), grp in sub.groupby(
            ["split_regime", "modelo", "chart", "k"], observed=True):
        val = grp.dropna(subset=["y_true", "y_pred"])
        if val.empty:
            continue
        linhas.append({
            "split_regime": regime, "recorte": recorte, "modelo": modelo, "chart": chart, "k": k,
            "rmse": float(np.sqrt(np.mean((val.y_true - val.y_pred) ** 2))),
            "acerto_direcional": directional_accuracy(
                val.y_origem.to_numpy(), val.y_true.to_numpy(), val.y_pred.to_numpy()),
            "n_origens": len(val), "n_musicas": int(val.song_id.nunique()),
            "n_excluidas": int(len(grp) - len(val)),
        })
    return pd.DataFrame(linhas)

tabela = pd.concat([celulas(tudo, r) for r in RECORTES], ignore_index=True)

# desvio entre seeds dos modelos neurais, vindo do item 09
p_resumo = MATRIZ / "resumo_mode2.parquet"
if p_resumo.exists():
    disp = pd.read_parquet(p_resumo).query("orcamento_arestas == 'completo'")
    disp = disp.rename(columns={"variante": "modelo", "rmse_desvio": "rmse_desvio_seeds"})
    tabela = tabela.merge(
        disp[["modelo", "split_regime", "recorte", "chart", "k", "rmse_desvio_seeds", "n_seeds"]],
        on=["modelo", "split_regime", "recorte", "chart", "k"], how="left")
tabela["rmse_desvio_seeds"] = tabela["rmse_desvio_seeds"].fillna(0.0)
tabela.to_parquet(OUT / "tabela_comparacao.parquet", index=False)

principal = tabela[tabela.recorte == "onchart"]
display(principal.pivot_table(index=["split_regime", "chart", "k"], columns="modelo",
                              values="rmse").reindex(columns=MODELOS).round(6))


In [ ]:
# A mesma tabela na leitura completa, que é a de referência e não a principal
display(tabela[tabela.recorte == "full"]
        .pivot_table(index=["split_regime", "chart", "k"], columns="modelo", values="rmse")
        .reindex(columns=MODELOS).round(6))

# Acerto direcional, leitura principal
display(principal.pivot_table(index=["split_regime", "chart", "k"], columns="modelo",
                              values="acerto_direcional").reindex(columns=MODELOS).round(3))


## 4. A estatística (item 03)

Cada comparação é `proposta` contra um dos outros quatro, na mesma célula. O erro quadrático de
cada origem é promediado **por música** antes do teste: origens da mesma música não são
observações independentes, e parear direto nas ~23 mil origens em vez das ~1,9 mil músicas infla a
significância.

A correção de Holm é aplicada dentro de cada família `(recorte, regime de split)`, que é o conjunto
de comparações lido junto: 4 modelos × 2 charts × 3 horizontes.

`dif_media < 0` significa erro **menor** da proposta.


In [ ]:
linhas = []
for recorte in RECORTES:
    sub_r = tudo if recorte == "full" else tudo[tudo.onchart]
    for regime in sorted(sub_r.split_regime.unique()):
        for chart in ("viral50", "top200"):
            for k in KS:
                base = sub_r[(sub_r.split_regime == regime) & (sub_r.chart == chart)
                             & (sub_r.k == k)].dropna(subset=["y_true", "y_pred"])
                prop = base[base.modelo == "proposta"].set_index(["song_id", "origin_week"])
                if prop.empty:
                    continue
                for outro in [m for m in MODELOS if m != "proposta"]:
                    alt = base[base.modelo == outro].set_index(["song_id", "origin_week"])
                    par = prop[["y_true", "y_pred"]].join(
                        alt[["y_pred"]], how="inner", lsuffix="_p", rsuffix="_o")
                    if len(par) < 2:
                        continue
                    e_prop = (par.y_true - par.y_pred_p) ** 2
                    e_outro = (par.y_true - par.y_pred_o) ** 2
                    r = paired_by_song(e_prop.to_numpy(), e_outro.to_numpy(),
                                       par.index.get_level_values("song_id").to_numpy(),
                                       seed=SEED_BOOTSTRAP)
                    linhas.append({"recorte": recorte, "split_regime": regime, "chart": chart,
                                   "k": k, "contra": outro, **r})

testes = pd.DataFrame(linhas)
for (recorte, regime), grp in testes.groupby(["recorte", "split_regime"]):
    testes.loc[grp.index, "p_holm"] = holm_correction(grp["p_value"].to_numpy())
    testes.loc[grp.index, "familia_n"] = len(grp)

testes["proposta_vence"] = (testes.ci_hi < 0) & (testes.p_holm < ALFA)
testes["proposta_perde"] = (testes.ci_lo > 0) & (testes.p_holm < ALFA)
testes.to_parquet(OUT / "testes_pareados.parquet", index=False)

display(testes[testes.recorte == "onchart"][
    ["split_regime", "contra", "chart", "k", "mean_diff", "ci_lo", "ci_hi",
     "p_value", "p_holm", "win_rate", "n_songs", "proposta_vence"]].round(6))


## 5. Ordenação por regime de split

O ticket 10 pede a ordenação reportada **separadamente para cada regime**: se ela mudar entre o
split atual e o pré-pandemia, o resultado do split atual não pode ser lido como propriedade do
modelo, e sim como propriedade do período
([ADR-0005](../docs/adr/0005-segundo-split-pre-pandemia.md)).


In [ ]:
ordem = (tabela[tabela.recorte == "onchart"]
         .groupby(["split_regime", "modelo"])["rmse"].mean()
         .reset_index()
         .sort_values(["split_regime", "rmse"]))
for regime, grp in ordem.groupby("split_regime"):
    print(f"{regime}: " + "  <  ".join(
        f"{r.modelo} ({r.rmse:.5f})" for r in grp.itertuples()))

fig, ax = plt.subplots(figsize=(9, 4.5))
larg = 0.8 / max(1, ordem.split_regime.nunique())
for i, (regime, grp) in enumerate(ordem.groupby("split_regime")):
    g = grp.set_index("modelo").reindex(MODELOS)
    ax.bar(np.arange(len(MODELOS)) + i * larg, g["rmse"], width=larg, label=regime)
ax.set_xticks(np.arange(len(MODELOS)) + larg / 2, MODELOS, rotation=15)
ax.set_ylabel("RMSE médio (on-chart, média dos 6 cenários)"); ax.legend()
fig.tight_layout(); plt.show()


## 6. O pré-compromisso, acionado

A regra é a do [ADR-0001](../docs/adr/0001-precompromisso-de-falseamento.md), aplicada à leitura
on-chart, por regime de split. "Vencer" um comparado significa: IC bootstrap da diferença
inteiramente abaixo de zero e `p` corrigido por Holm abaixo de 0,05, na **maioria** das seis
células (2 charts × 3 horizontes).

Os três desfechos previstos e o que a dissertação passa a afirmar em cada um estão no ADR. A
célula abaixo apenas conta e escolhe: a interpretação já estava escrita.


In [ ]:
N_CELULAS = len(KS) * 2
veredito = []
for regime in sorted(testes.split_regime.unique()):
    sub = testes[(testes.recorte == "onchart") & (testes.split_regime == regime)]
    v_sem = int(sub[(sub.contra == "sem_grafo")].proposta_vence.sum())
    v_emb = int(sub[(sub.contra == "embaralhado")].proposta_vence.sum())
    maioria = N_CELULAS // 2 + 1
    if v_sem >= maioria and v_emb >= maioria:
        caso = "A tese se sustenta como está: o ganho é atribuível à topologia."
    elif v_sem >= maioria:
        caso = ("O ganho vem de agregar atributos de vizinhos, não da topologia. "
                "A tese é reescrita nesses termos.")
    else:
        caso = ("A tese vira condicional: o trabalho entrega o mapa de ONDE a estrutura paga "
                "e reporta equivalência no resto.")
    veredito.append({"split_regime": regime, "vence_sem_grafo": f"{v_sem}/{N_CELULAS}",
                     "vence_embaralhado": f"{v_emb}/{N_CELULAS}", "desfecho": caso})
veredito = pd.DataFrame(veredito)
display(veredito)
veredito.to_parquet(OUT / "veredito_adr0001.parquet", index=False)

if veredito.desfecho.nunique() > 1:
    print("\n⚠️ Os dois regimes de split levam a desfechos diferentes. Isto não é ruído a "
          "resolver:\né o resultado do ADR-0005, e entra na discussão como tal.")


## 7. O que levar de volta

Do Drive (`comparacao_final/`):

- `predicoes_cinco_modelos.parquet` — a base de tudo, uma linha por (música, chart, origem, k, modelo);
- `tabela_comparacao.parquet` — a tabela do capítulo de resultados;
- `testes_pareados.parquet` — IC da diferença, `p` bruto e `p` corrigido por Holm, por célula;
- `sir_mode2_{regime}.parquet` — o cache caro do SIR, para não refazer;
- `veredito_adr0001.parquet` — o desfecho declarado por regime de split.

Depois disso: escrever o capítulo de resultados a partir da tabela e do veredito, e atualizar o
[ROADMAP](../ROADMAP.md). Os itens 18 e 19 (cotrajetória por chart, cabeça sensível ao chart) e o
item 15 (arquitetura com atenção) são variações de arquitetura, e só fazem sentido depois desta
leitura.
